# NBS site query — landslide E2E (Porto Alegre)

Bairro-level landslide screening on the **90 m** OEF grid: slope / rain / soil / vegetation + slope-stabilization NBS.

| Section | Unit | Purpose |
|---------|------|---------|
| **Bairro** | Bairro polygon | Step 0 priority + mechanism + landslide NBS |
| **Grid** | 90 m cell | Per-cell landslide mechanism type + dominant NBS |

Follows [`recommended-datasets.md`](../docs/recommended-datasets.md) and [`landslide_nbs_dataset_lens.md`](../docs/landslide_nbs_dataset_lens.md).

**Default site:** Glória — hillslope bairro (slope ≥ 15° activation gate in hazard score)

**CLI:** `run_e2e.py --hazard landslide --site Glória`

**Setup:** run with cwd = `scripts/`; use nbs_e2e venv or floods `.venv`.


## Setup — paths and imports


In [ ]:
import json
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask
from rasterio.transform import array_bounds
from shapely.geometry import mapping

NOTEBOOK_DIR = Path.cwd()
NBS_E2E_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "scripts" else NOTEBOOK_DIR
SCRIPTS_DIR = NBS_E2E_ROOT / "scripts"
if not (SCRIPTS_DIR / "catalog_layers.py").exists():
    raise FileNotFoundError(
        "Run this notebook with cwd = transformation/nbs_screening/scripts "
        f"(expected catalog_layers.py under {SCRIPTS_DIR})"
    )
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

for mod in ("catalog_layers", "nbs_rules", "grid_screening"):
    sys.modules.pop(mod, None)
import catalog_layers as _catalog_layers
import nbs_rules as _nbs_rules

OUT_DIR = NBS_E2E_ROOT / "output"
IN_DIR = OUT_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

HAZARD = "landslide"
CATALOG_COGS = _catalog_layers.get_catalog_cogs(HAZARD)
LOCAL_SCREENING_RASTERS = _catalog_layers.get_local_rasters(HAZARD)
barrio_landslide_context = _catalog_layers.barrio_landslide_context
query_layers = _catalog_layers.query_layers
recommend_all = _nbs_rules.recommend_all
LANDSLIDE_NBS_TYPES = _nbs_rules.LANDSLIDE_NBS_TYPES

import grid_screening as _grid_screening
screen_bairro_grid = _grid_screening.screen_bairro_grid
screen_poa_landslide_mechanism_grid = _grid_screening.screen_poa_landslide_mechanism_grid
cells_to_geodataframe = _grid_screening.cells_to_geodataframe
result_to_geojson = _grid_screening.result_to_geojson
export_landslide_mechanism_geotiff = _grid_screening.export_landslide_mechanism_geotiff
export_poa_landslide_mechanism_layers = _grid_screening.export_poa_landslide_mechanism_layers
MECHANISM_RASTER_NODATA = _grid_screening.MECHANISM_RASTER_NODATA

LANDSLIDE_MECHANISM_TYPE_CODES = _nbs_rules.LANDSLIDE_MECHANISM_TYPE_CODES
MECHANISM_MIXED_COLOR = _nbs_rules.MECHANISM_MIXED_COLOR
LANDSLIDE_MECHANISM_CATALOG_DOCS = _nbs_rules.LANDSLIDE_MECHANISM_CATALOG_DOCS
LANDSLIDE_MECHANISM_DISPLAY_LABELS = _nbs_rules.LANDSLIDE_MECHANISM_DISPLAY_LABELS
LANDSLIDE_MIN_STRENGTH = _nbs_rules.LANDSLIDE_MIN_STRENGTH

LANDSLIDE_MECHANISM_COLORS = {
    "without_clear_dominant": "#e0e0e0",
    "steep_activatable_slope": "#b2182b",
    "rainfall_trigger": "#2166ac",
    "low_cohesion_wet": "#8c510a",
    "vegetation_deficit": "#d8b365",
    "drainage_saturation": "#4393c3",
    "disturbed_bare_slope": "#a6611a",
    "upslope_convergence": "#5e3c99",
    "high_social_exposure": "#636363",
    "mixed": MECHANISM_MIXED_COLOR,
}
GRID_LAYER_IDS = {"app_landslide_90m", "sample_grid_1km"}

LANDSLIDE_HAZARD = HAZARD
LANDSLIDE_SITE_NAME = "Glória"  # change to test another bairro
print("NBS E2E root:", NBS_E2E_ROOT)
print("Hazard:", HAZARD)
print("Site:", LANDSLIDE_SITE_NAME)
print("Catalog layers:", list(CATALOG_COGS.keys()))
print("Local rasters:", list(LOCAL_SCREENING_RASTERS.keys()))


## Step 0 — Landslide priority screening

In [ ]:
landslide_ctx_row = barrio_landslide_context(LANDSLIDE_SITE_NAME)
landslide_site_geom = landslide_ctx_row.pop("geometry")

landslide_site_gdf = gpd.GeoDataFrame(
    [{"name": LANDSLIDE_SITE_NAME, "geometry": landslide_site_geom}], crs="EPSG:4326"
)
landslide_site_path = IN_DIR / f"site_{LANDSLIDE_SITE_NAME.lower().replace(' ', '_')}.geojson"
landslide_site_gdf.to_file(landslide_site_path, driver="GeoJSON")

landslide_step0 = pd.Series(
    {
        "bairro": LANDSLIDE_SITE_NAME,
        "hazard_mean": landslide_ctx_row["hazard_mean"],
        "risk_mean": landslide_ctx_row["risk_mean"],
        "exposure_score": landslide_ctx_row["exposure_score"],
        "vulnerability_score": landslide_ctx_row["vulnerability_score"],
    }
)
display(landslide_step0.to_frame("value"))
print(f"Site saved → {landslide_site_path}")


## Step 1a — Query landslide diagnostic layers

In [ ]:
landslide_layers = query_layers(landslide_site_geom, hazard=LANDSLIDE_HAZARD)

landslide_rows = []
for layer in landslide_layers:
    row = {
        "layer_id": layer.layer_id,
        "status": layer.status,
        "note": layer.note[:80] if layer.note else "",
    }
    for k, v in layer.stats.items():
        row[k] = round(v, 4) if isinstance(v, float) else v
    landslide_rows.append(row)

display(pd.DataFrame(landslide_rows))


### Inspect landslide screening metrics

In [ ]:
landslide_grid_layer = next(
    l for l in landslide_layers if l.layer_id in GRID_LAYER_IDS
)
landslide_water_layer = next(l for l in landslide_layers if l.layer_id == "osm_waterways")

landslide_grid_stats = landslide_grid_layer.stats
landslide_water_stats = {**landslide_water_layer.stats, "_note": landslide_water_layer.note}

print("Screening pixels:", landslide_grid_stats.get("n_cells"))
display(
    pd.Series({k: v for k, v in landslide_grid_stats.items() if k != "n_cells"}).to_frame("mean")
)


## Step 1b — Infer landslide susceptibility context

In [ ]:
landslide_ctx = {"bairro": LANDSLIDE_SITE_NAME, "hazard": LANDSLIDE_HAZARD, **landslide_ctx_row}

landslide_mechanism, landslide_recommendations = recommend_all(
    landslide_ctx, landslide_grid_stats, landslide_water_stats, hazard=LANDSLIDE_HAZARD
)

landslide_mech_df = pd.DataFrame(
    {
        "signal": [
            "steep_activatable_slope",
            "rainfall_trigger",
            "low_cohesion_wet",
            "vegetation_deficit",
            "drainage_saturation",
            "disturbed_bare_slope",
            "upslope_convergence",
            "high_social_exposure",
        ],
        "value": [
            landslide_mechanism.steep_activatable_slope,
            landslide_mechanism.rainfall_trigger,
            landslide_mechanism.low_cohesion_wet,
            landslide_mechanism.vegetation_deficit,
            landslide_mechanism.drainage_saturation,
            landslide_mechanism.disturbed_bare_slope,
            landslide_mechanism.upslope_convergence,
            landslide_mechanism.high_social_exposure,
        ],
    }
)
display(landslide_mech_df)
for line in landslide_mechanism.rationale:
    print(" •", line)


## Step 2 — Landslide NBS typology screening

In [ ]:
landslide_recs_df = pd.DataFrame([asdict(r) for r in landslide_recommendations])
landslide_recs_df["gaps"] = landslide_recs_df["gaps"].apply(lambda g: "; ".join(g) if g else "")
display(landslide_recs_df[["nbs_type", "score", "rationale", "gaps"]])


## Save landslide report (JSON)

In [ ]:
landslide_report = {
    "exercise": "nbs_site_query_landslide_e2e",
    "hazard": LANDSLIDE_HAZARD,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "site": {"name": LANDSLIDE_SITE_NAME, "geojson": str(landslide_site_path)},
    "step_0_priority": landslide_ctx,
    "step_1_mechanism": {
        "steep_activatable_slope": landslide_mechanism.steep_activatable_slope,
        "rainfall_trigger": landslide_mechanism.rainfall_trigger,
        "low_cohesion_wet": landslide_mechanism.low_cohesion_wet,
        "vegetation_deficit": landslide_mechanism.vegetation_deficit,
        "drainage_saturation": landslide_mechanism.drainage_saturation,
        "disturbed_bare_slope": landslide_mechanism.disturbed_bare_slope,
        "upslope_convergence": landslide_mechanism.upslope_convergence,
        "high_social_exposure": landslide_mechanism.high_social_exposure,
        "rationale": landslide_mechanism.rationale,
    },
    "step_2_nbs_recommendations": [asdict(r) for r in landslide_recommendations],
}

landslide_out = OUT_DIR / f"nbs_site_query_landslide_{LANDSLIDE_SITE_NAME.lower().replace(' ', '_')}.json"
landslide_out.write_text(json.dumps(landslide_report, indent=2, ensure_ascii=False))
print(f"Report written → {landslide_out}")


## Grid screening — landslide mechanism type layer

Per **90 m** cell inside the bairro (not only bairro-mean aggregates):

| Field | Meaning |
|-------|---------|
| `landslide_mechanism_type` | Dominant type: `steep_activatable_slope` · `rainfall_trigger` · `low_cohesion_wet` · `vegetation_deficit` · `drainage_saturation` · `disturbed_bare_slope` · `upslope_convergence` · `high_social_exposure` · `mixed` · `without_clear_dominant` |
| `landslide_mechanism_code` | Integer code for catalog raster (0–9) |
| Boolean flags | Eight mechanism signals (can overlap; type picks dominant by strength ≥ 0.15) |

Logic: `nbs_rules.classify_dominant_landslide_mechanism()`.

In [ ]:
landslide_grid_result = screen_bairro_grid(
    LANDSLIDE_SITE_NAME,
    hazard=LANDSLIDE_HAZARD,
    sample_catalog=True,
    include_nbs=False,
    require_positive_hazard=True,
    preload_layers=True,
    zonal_fallback=False,
)
landslide_grid_gdf = cells_to_geodataframe(landslide_grid_result)

display(pd.Series(landslide_grid_result.mechanism_summary).to_frame("value"))
if "dominant_mechanism_type_counts" in landslide_grid_result.mechanism_summary:
    display(
        pd.Series(landslide_grid_result.mechanism_summary["dominant_mechanism_type_counts"])
        .sort_values(ascending=False)
        .to_frame("cells")
    )

display(
    landslide_grid_gdf[
        [
            "cell_id",
            "landslide_mechanism_type",
            "landslide_mechanism_code",
            "landslide_score",
            "slope_deg",
            "steep_activatable_slope",
            "rainfall_trigger",
            "vegetation_deficit",
        ]
    ].head(12)
)

### Optional — full POA landslide mechanism layer

Set `BUILD_POA_LANDSLIDE_LAYER = True` to build the city-wide landslide mechanism raster (~3.7k hazard-active 90 m pixels; **~20 s** with layer preload). **Restart the kernel** after pulling code changes so `grid_screening.py` reloads.

**Methodology:** [`docs/poa_mechanism_type_layer.md`](../docs/poa_mechanism_type_layer.md)

| Output | Description |
|--------|-------------|
| `landslide_mechanism_type_poa_90m_observed.tif` | Direct screening on hazard-active pixels (H > 0) |
| `landslide_mechanism_type_poa_90m.tif` | **Filled** layer (observed + IDW in gaps) |
| `landslide_mechanism_is_interpolated_poa_90m.tif` | Mask: 1 = IDW-filled pixel |
| `landslide_mechanism_type_poa_90m.geojson` | Screened cells (+ strengths, `mixed_tied_mechanisms`, `is_interpolated`) |

Then **`PUBLISH_POA_LANDSLIDE_COG_TILES = True`** (cell below) to build COG + tiles and upload to S3.

In [ ]:
BUILD_POA_LANDSLIDE_LAYER = True  # flip True to export full POA rasters + GeoJSON

if not BUILD_POA_LANDSLIDE_LAYER:
    print("Skipping POA landslide layer (BUILD_POA_LANDSLIDE_LAYER=False).")
else:
    poa_landslide_result = screen_poa_landslide_mechanism_grid(
        sample_catalog=True,
        include_nbs=False,
    )
    poa_landslide_paths = export_poa_landslide_mechanism_layers(poa_landslide_result, OUT_DIR)

    poa_landslide_geojson = OUT_DIR / "landslide_mechanism_type_poa_90m.geojson"
    poa_landslide_payload = result_to_geojson(poa_landslide_result)
    poa_landslide_payload["properties"]["layer"] = "poa_landslide_mechanism_type"
    poa_landslide_payload["properties"]["mechanism_type_codes"] = LANDSLIDE_MECHANISM_TYPE_CODES
    poa_landslide_payload["properties"]["mechanism_summary"] = {
        **poa_landslide_result.mechanism_summary,
        "hazard_valid_cell_count": sum(1 for c in poa_landslide_result.cells if c.hazard_valid),
        "interpolated_cell_count": sum(1 for c in poa_landslide_result.cells if c.is_interpolated),
    }
    poa_landslide_geojson.write_text(json.dumps(poa_landslide_payload), encoding="utf-8")

    display(
        pd.Series(poa_landslide_result.mechanism_summary.get("dominant_mechanism_type_counts", {}))
        .sort_values(ascending=False)
        .to_frame("cells")
    )
    print("POA observed TIF →", poa_landslide_paths["observed"])
    print("POA filled TIF   →", poa_landslide_paths["filled"])
    print("POA interp mask  →", poa_landslide_paths["is_interpolated"])
    print("POA GeoJSON      →", poa_landslide_geojson)

### Map — full POA `landslide_mechanism_type`

Visualizes `output/landslide_mechanism_type_poa_90m.tif` (preferred) or `.geojson` after `BUILD_POA_LANDSLIDE_LAYER = True`.

In [ ]:
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch
from rasterio.plot import plotting_extent

POA_LANDSLIDE_TIF = OUT_DIR / "landslide_mechanism_type_poa_90m.tif"
POA_LANDSLIDE_GEOJSON = OUT_DIR / "landslide_mechanism_type_poa_90m.geojson"

LANDSLIDE_TYPE_ORDER = list(LANDSLIDE_MECHANISM_TYPE_CODES.keys())
POA_LANDSLIDE_CMAP = ListedColormap(
    [LANDSLIDE_MECHANISM_COLORS[t] for t in LANDSLIDE_TYPE_ORDER]
)
POA_LANDSLIDE_NORM = BoundaryNorm(
    np.arange(-0.5, len(LANDSLIDE_TYPE_ORDER) + 0.5, 1), POA_LANDSLIDE_CMAP.N
)

fig, ax = plt.subplots(figsize=(11, 10))
if POA_LANDSLIDE_TIF.exists():
    with rasterio.open(POA_LANDSLIDE_TIF) as src:
        data = src.read(1, masked=True)
        if src.nodata != MECHANISM_RASTER_NODATA:
            data = np.ma.masked_equal(data, MECHANISM_RASTER_NODATA)
        ax.imshow(
            data,
            extent=plotting_extent(src),
            origin="upper",
            cmap=POA_LANDSLIDE_CMAP,
            norm=POA_LANDSLIDE_NORM,
            interpolation="nearest",
        )
    source_label = POA_LANDSLIDE_TIF.name
elif POA_LANDSLIDE_GEOJSON.exists():
    poa_landslide_gdf = gpd.read_file(POA_LANDSLIDE_GEOJSON)
    poa_landslide_gdf["color"] = (
        poa_landslide_gdf["landslide_mechanism_type"]
        .map(LANDSLIDE_MECHANISM_COLORS)
        .fillna("#cccccc")
    )
    poa_landslide_gdf.plot(ax=ax, color=poa_landslide_gdf["color"], edgecolor="none", alpha=0.95)
    source_label = POA_LANDSLIDE_GEOJSON.name
else:
    raise FileNotFoundError(
        "Run BUILD_POA_LANDSLIDE_LAYER=True or ensure output/landslide_mechanism_type_poa_90m.tif exists."
    )

try:
    landslide_site_gdf.boundary.plot(
        ax=ax,
        color="black",
        linewidth=1.2,
        linestyle="--",
        label=LANDSLIDE_SITE_NAME,
    )
except NameError:
    site_name = globals().get("LANDSLIDE_SITE_NAME", "Glória")
    site_path = IN_DIR / f"site_{site_name.lower().replace(' ', '_')}.geojson"
    if site_path.exists():
        gpd.read_file(site_path).boundary.plot(
            ax=ax,
            color="black",
            linewidth=1.2,
            linestyle="--",
            label=site_name,
        )

ax.set_title(f"Dominant landslide mechanism type — Porto Alegre (90 m)\\nsource: {source_label}")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
legend_handles = [
    Patch(
        facecolor=LANDSLIDE_MECHANISM_COLORS[t],
        edgecolor="white",
        label=LANDSLIDE_MECHANISM_DISPLAY_LABELS.get(t, t),
    )
    for t in LANDSLIDE_TYPE_ORDER
]
ax.legend(handles=legend_handles, loc="lower left", fontsize=7, title="landslide_mechanism_type")
plt.tight_layout()
plt.show()

### Publish — COG + tiles + GeoJSON to S3

After `BUILD_POA_LANDSLIDE_LAYER = True`, set `PUBLISH_POA_LANDSLIDE_COG_TILES = True` to build EPSG:3857 COG + XYZ tiles locally, then upload **COG, tiles, and screened-cell GeoJSON** to S3 (`UPLOAD_POA_TO_S3`).

In [ ]:
import os
import shutil
import subprocess

PUBLISH_POA_LANDSLIDE_COG_TILES = True  # flip True after BUILD_POA_LANDSLIDE_LAYER produces the .tif
UPLOAD_POA_TO_S3 = True  # requires AWS CLI (aws configure)

POA_LANDSLIDE_LAYER_SLUG = "landslide_mechanism_type_poa_90m"
IN_LANDSLIDE_TIF = OUT_DIR / f"{POA_LANDSLIDE_LAYER_SLUG}.tif"
POA_LANDSLIDE_PUBLISH_DIR = OUT_DIR / POA_LANDSLIDE_LAYER_SLUG
LANDSLIDE_COLORS_TXT = POA_LANDSLIDE_PUBLISH_DIR / f"{POA_LANDSLIDE_LAYER_SLUG}_colors.txt"
LANDSLIDE_WARPED_TIF = POA_LANDSLIDE_PUBLISH_DIR / f"{POA_LANDSLIDE_LAYER_SLUG}_3857.tif"
LANDSLIDE_COG_TIF = POA_LANDSLIDE_PUBLISH_DIR / f"{POA_LANDSLIDE_LAYER_SLUG}_cog.tif"
LANDSLIDE_COLORIZED_TIF = POA_LANDSLIDE_PUBLISH_DIR / f"{POA_LANDSLIDE_LAYER_SLUG}_colorized.tif"
LANDSLIDE_VALUE_RGB_TIF = POA_LANDSLIDE_PUBLISH_DIR / f"{POA_LANDSLIDE_LAYER_SLUG}_value_encoded_rgb.tif"
LANDSLIDE_VISUAL_TILES_DIR = POA_LANDSLIDE_PUBLISH_DIR / "tiles_visual"
LANDSLIDE_VALUE_TILES_DIR = POA_LANDSLIDE_PUBLISH_DIR / "tiles_values"
LANDSLIDE_VALUE_DECODE_TXT = POA_LANDSLIDE_PUBLISH_DIR / f"{POA_LANDSLIDE_LAYER_SLUG}_value_tiles_decode.txt"

_GDAL_CLI = (
    "gdalwarp",
    "gdal_translate",
    "gdaldem",
    "gdal_calc.py",
    "gdal2tiles.py",
)


def _hex_to_rgb(hex_color: str) -> tuple[int, int, int]:
    h = hex_color.lstrip("#")
    return tuple(int(h[i : i + 2], 16) for i in (0, 2, 4))


def _prepend_common_bin_to_path() -> None:
    for prefix in (Path("/opt/homebrew/bin"), Path("/usr/local/bin")):
        if prefix.is_dir():
            p = str(prefix)
            if p not in os.environ.get("PATH", "").split(":"):
                os.environ["PATH"] = f"{p}:{os.environ.get('PATH', '')}"


def _require_gdal_cli() -> None:
    _prepend_common_bin_to_path()
    missing = [cmd for cmd in _GDAL_CLI if not shutil.which(cmd)]
    if missing:
        raise RuntimeError(
            f"Missing GDAL CLI tools: {missing}. "
            "Install GDAL (e.g. brew install gdal) and restart the kernel."
        )


if not PUBLISH_POA_LANDSLIDE_COG_TILES:
    print(
        "Skipping POA landslide COG/tiles publish (PUBLISH_POA_LANDSLIDE_COG_TILES=False). "
        "Set True after landslide_mechanism_type_poa_90m.tif exists."
    )
elif not IN_LANDSLIDE_TIF.exists():
    raise FileNotFoundError(
        f"Missing input raster: {IN_LANDSLIDE_TIF}. Run BUILD_POA_LANDSLIDE_LAYER=True first."
    )
else:
    _require_gdal_cli()
    POA_LANDSLIDE_PUBLISH_DIR.mkdir(parents=True, exist_ok=True)

    color_lines = [
        "# Dominant landslide mechanism type. GDAL color-relief for visual tiles.",
        "# Codes from LANDSLIDE_MECHANISM_TYPE_CODES in nbs_rules.py",
        f"# Raster nodata = {MECHANISM_RASTER_NODATA}; code 0 = without_clear_dominant",
        "nv 0 0 0 0",
    ]
    for mech_type, code in sorted(LANDSLIDE_MECHANISM_TYPE_CODES.items(), key=lambda kv: kv[1]):
        r, g, b = _hex_to_rgb(LANDSLIDE_MECHANISM_COLORS[mech_type])
        color_lines.append(f"{code} {r} {g} {b}")
    LANDSLIDE_COLORS_TXT.write_text("\n".join(color_lines) + "\n", encoding="utf-8")
    print("Wrote colors:", LANDSLIDE_COLORS_TXT)

    subprocess.run(
        [
            "gdalwarp",
            "-t_srs",
            "EPSG:3857",
            "-r",
            "near",
            "-overwrite",
            str(IN_LANDSLIDE_TIF),
            str(LANDSLIDE_WARPED_TIF),
        ],
        check=True,
    )
    subprocess.run(
        [
            "gdal_translate",
            str(LANDSLIDE_WARPED_TIF),
            str(LANDSLIDE_COG_TIF),
            "-of",
            "COG",
            "-ot",
            "Byte",
            "-co",
            "COMPRESS=DEFLATE",
            "-co",
            "RESAMPLING=NEAREST",
            "-co",
            "OVERVIEWS=AUTO",
        ],
        check=True,
    )
    print("Created COG:", LANDSLIDE_COG_TIF)

    subprocess.run(
        [
            "gdaldem",
            "color-relief",
            "-nearest_color_entry",
            str(LANDSLIDE_COG_TIF),
            str(LANDSLIDE_COLORS_TXT),
            str(LANDSLIDE_COLORIZED_TIF),
            "-alpha",
        ],
        check=True,
    )
    print("Created colorized raster:", LANDSLIDE_COLORIZED_TIF)

    LANDSLIDE_VISUAL_TILES_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "gdal2tiles.py",
            "-r",
            "near",
            "-z",
            "8-15",
            "--xyz",
            "-w",
            "none",
            str(LANDSLIDE_COLORIZED_TIF),
            str(LANDSLIDE_VISUAL_TILES_DIR),
        ],
        check=True,
    )
    print("Visual tiles:", LANDSLIDE_VISUAL_TILES_DIR)

    base_expr = (
        "numpy.where(numpy.isnan(A), 0, "
        "numpy.rint(numpy.clip(A,0,16777214)).astype(numpy.int64) + 1)"
    )
    subprocess.run(
        [
            "gdal_calc.py",
            "-A",
            str(LANDSLIDE_COG_TIF),
            "--calc",
            f"bitwise_and({base_expr},255)",
            "--calc",
            f"bitwise_and(right_shift({base_expr},8),255)",
            "--calc",
            f"bitwise_and(right_shift({base_expr},16),255)",
            "--type",
            "Byte",
            "--NoDataValue",
            "0",
            "--overwrite",
            "--outfile",
            str(LANDSLIDE_VALUE_RGB_TIF),
        ],
        check=True,
    )

    LANDSLIDE_VALUE_TILES_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "gdal2tiles.py",
            "-r",
            "near",
            "-z",
            "8-15",
            "--xyz",
            "-w",
            "none",
            str(LANDSLIDE_VALUE_RGB_TIF),
            str(LANDSLIDE_VALUE_TILES_DIR),
        ],
        check=True,
    )
    print("Value tiles:", LANDSLIDE_VALUE_TILES_DIR)

    LANDSLIDE_VALUE_DECODE_TXT.write_text(
        "\n".join(
            [
                "Landslide mechanism type POA value tiles",
                "",
                f"Source raster: {IN_LANDSLIDE_TIF}",
                f"COG (EPSG:3857): {LANDSLIDE_COG_TIF}",
                f"Visual tiles: {LANDSLIDE_VISUAL_TILES_DIR}/{{z}}/{{x}}/{{y}}.png",
                f"Value tiles: {LANDSLIDE_VALUE_TILES_DIR}/{{z}}/{{x}}/{{y}}.png",
                "",
                "Value tile encoding (Terrain RGB style):",
                "encoded = R + 256 * G + 65536 * B",
                "if encoded == 0: nodata",
                "else: landslide_mechanism_code = encoded - 1",
                "",
                "Mechanism codes:",
                *[
                    f"  {code}: {mech_type}"
                    for mech_type, code in sorted(
                        LANDSLIDE_MECHANISM_TYPE_CODES.items(), key=lambda kv: kv[1]
                    )
                ],
            ]
        )
        + "\n",
        encoding="utf-8",
    )
    print("Value decode notes:", LANDSLIDE_VALUE_DECODE_TXT)

    if UPLOAD_POA_TO_S3:
        import poa_mechanism_publish as _poa_publish

        _poa_publish.upload_poa_mechanism_to_s3(
            "landslide",
            OUT_DIR,
            publish_dir=POA_LANDSLIDE_PUBLISH_DIR,
            geojson_path=OUT_DIR / f"{POA_LANDSLIDE_LAYER_SLUG}.geojson",
        )
    else:
        print("Skipping S3 upload (UPLOAD_POA_TO_S3=False).")

---

### Try another site

Change `LANDSLIDE_SITE_NAME` in Setup.

```bash
geospatial-data/floods/.venv/bin/python transformation/nbs_screening/scripts/run_e2e.py --hazard landslide --site Glória
```


In [ ]:
import geopandas as gpd
import pandas as pd

poa = gpd.read_file(OUT_DIR / "landslide_mechanism_type_poa_90m.geojson")
mixed = poa[poa["landslide_mechanism_type"] == "mixed"].copy()

flag_cols = [
    "steep_activatable_slope", "rainfall_trigger", "low_cohesion_wet",
    "vegetation_deficit", "drainage_saturation", "disturbed_bare_slope",
    "upslope_convergence", "high_social_exposure",
]
mixed["active_flags"] = mixed[flag_cols].apply(
    lambda r: ", ".join(c for c in flag_cols if r[c]), axis=1
)

display(mixed["active_flags"].value_counts().head(10).to_frame("cells"))
display(mixed[["cell_id", "slope_deg", "landslide_score", "active_flags"]].head(20))